<a href="https://colab.research.google.com/github/gauravd12345/wikiGPT/blob/main/wikiGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
!pip install convokit
from convokit import Corpus, download

In [ ]:
corpus = Corpus(filename=download("movie-corpus")) # takes ~1-2 minutes

In [ ]:
corpus.print_summary_stats()

In [ ]:
utt = corpus.random_utterance()

print("ID:", utt.id, "\n")
print("Reply_to:", utt.reply_to, "\n")
print("Timestamp:", utt.timestamp, "\n")
print("Text:", utt.text, "\n")
print("Conversation ID:", utt.conversation_id, "\n")
print("Speaker ID:", utt.speaker.id)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"eos_token": "<eos>", "additional_special_tokens": ["<sep>"]})

In [ ]:
def get_ordered_utterances(convo):
    utts = {utt.id: utt for utt in convo.iter_utterances()}
    roots = [utt for utt in utts.values() if utt.reply_to is None]
    if not roots:
        return list(utts.values())
    ordered = []
    current = roots[0]
    visited = set()
    while current and current.id not in visited:
        ordered.append(current)
        visited.add(current.id)
        current = next((u for u in utts.values() if u.reply_to == current.id), None)
    return ordered


buffer = []

for convo in corpus.iter_conversations():
    utterances = get_ordered_utterances(convo)
    texts = [utt.text.strip() for utt in utterances if utt.text and utt.text.strip()]
    if len(texts) < 2:
        continue
    buffer.extend(tokenizer.encode(" <sep> ".join(texts) + " <eos>"))

block_size = 64
chunks = [buffer[i : i + block_size + 1] for i in range(0, len(buffer) - block_size, block_size)]


X = torch.tensor([c[:-1] for c in chunks], dtype=torch.long)
y = torch.tensor([c[1:]  for c in chunks], dtype=torch.long)

print(X.shape, y.shape)

In [ ]:
text = tokenizer.decode(X[1])
text2 = tokenizer.decode(y[1])
print(text)
print(text2)

In [ ]:
""" Hyper parameter """

d_model = 512
h = 2
d_k = 64
d_v = 64
N = 2

lr = 0.001
epochs = 10
batch_size = 64
vocab_size = len(tokenizer)

In [ ]:
class DecoderTransformerDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

  def __len__(self):
    return len(self.X)

In [ ]:
class DecoderTransformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, d_model)
    self.pos = nn.Embedding(block_size, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)])
    self.W_v = nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)])

    self.W_o = nn.Linear(h * d_v, d_model) # final projection

    self.ffn1 = nn.Linear(d_model, 4 * d_model) # ffn layer
    self.ffn2 = nn.Linear(4 * d_model, d_model)

    self.fc = nn.Linear(d_model, vocab_size) # final layer

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def multi_head_attention(self, x):
    W_tot = []
    mask = torch.triu(torch.full((block_size, block_size), float('-inf')), diagonal=1).to(x.device)
    for Q, K, V in zip(self.W_q, self.W_k, self.W_v):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))         # query-key alignment
      alignment = alignment + mask                                 # adding mask

      wei = torch.softmax(alignment / (d_k ** 0.5), dim=1)         # alignment weights
      wei_value = torch.matmul(wei, V_i)                           # weighted values

      W_tot.append(wei_value)

    out = self.W_o(torch.cat(W_tot, dim=2))
    return out

  def forward(self, x): # (batch_size, block_size)
    p = torch.arange(block_size).to(x.device)
    x = self.embed(x) + self.pos(p)     # word & positional embedding

    for _ in range(N):
      out = self.multi_head_attention(x)  # multi head attention
      out = self.ln1(out + x)             # layernorm + residual connection

      fn = self.ffn1(out)                 # ffn
      fn = torch.relu(fn)
      fn = self.ffn2(fn)

      out = self.ln2(fn + out)
      x = out

    out = self.fc(out)
    return out


In [ ]:
dataset = DecoderTransformerDataset(X, y)
dataloader = DataLoader(
    dataset=dataset,
    batch_size=batch_size
)

model = DecoderTransformer().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

model.train()
for epoch in range(epochs):
  total_loss = 0.0
  pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
  for xb, yb in pbar:
    optimizer.zero_grad()

    xb = xb.to(device)
    yb = yb.to(device)

    out = model(xb)
    loss = criterion(out.transpose(1, 2), yb)
    total_loss += loss.item()

    loss.backward()
    optimizer.step()

    pbar.set_postfix(loss=f"{loss.item():.4f}")

  print(f"Epoch: {epoch} | Loss: {total_loss / batch_size}")


In [ ]:
test = "How are you doing?"
seq_len = 10

tokens = tokenizer.encode(test)

model.eval()
with torch.no_grad():
  for _ in range(seq_len):
      x = tokens[-block_size:]
      xb = torch.tensor(x).unsqueeze(0).to(device)
      out = model(xb)
      next_token = torch.argmax(out[0, -1], dim=-1).item()
      tokens.append(next_token)
      if next_token == tokenizer.eos_token_id:
          break

  print(tokenizer.decode(tokens))